# Train, evaluate, and extrapolate a SpecMine baseline

This notebook uses only specs with nonblank labels from the selected CSV for training and holdout evaluation. It keeps repositories disjoint for evaluation, then refits the selected Logistic Regression or Random Forest model on all labeled specs and predicts a random sample of specs whose IDs do not appear in the CSV. Blank-label rows are treated as already reviewed and are excluded from prediction. Predictions are model outputs, not verified authorship labels.

In [3]:
import logging
import sys
from pathlib import Path

project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "src" / "specmine").is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError(
        "Could not find src/specmine. Open this notebook from the project "
        "folder or one of its subfolders."
    )
sys.path.insert(0, str(project_root / "src"))
print(f"Notebook Python: {sys.executable}")
print(f"Project root: {project_root}")

import numpy as np
import pandas as pd
from sklearn.base import clone

from specmine.data import load_data, load_feature_data
from specmine.features import build_features
from specmine.labels import SPEC_ID_COLUMN, read_labels_csv
from specmine.train import (
    ID_COLUMN,
    REPOSITORY_COLUMN,
    prepare_training_data,
    train_models,
)

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logging.getLogger("specmine.train").setLevel(logging.INFO)

SEED = 42
TEST_SIZE = 0.2
MODEL_NAME = "logistic_regression"  # Choose "logistic_regression" or "random_forest".
PREDICTION_SAMPLE_SIZE = 100
LABELS_CSV = project_root / "examples" / "synthetic_labels.csv"
PREDICTIONS_CSV = project_root / "notebooks" / "unlabeled_predictions.csv"

print(f"Labels CSV: {LABELS_CSV}")
print(f"Prediction output: {PREDICTIONS_CSV}")

Notebook Python: /Users/austinburt/miniforge3/envs/tf-metal/bin/python
Project root: /Users/austinburt/Documents/Code/PythonEnv/Python/CS440/FinalProject-CS440
Labels CSV: /Users/austinburt/Documents/Code/PythonEnv/Python/CS440/FinalProject-CS440/examples/synthetic_labels.csv
Prediction output: /Users/austinburt/Documents/Code/PythonEnv/Python/CS440/FinalProject-CS440/notebooks/unlabeled_predictions.csv


## Load labeled data

Set `LABELS_CSV` to your reviewed labels before running. The supplied synthetic file is for smoke testing only: replace its fabricated rows before treating results as real. The full metadata table is used to match IDs, but feature rows are loaded only for nonblank labeled specs at this stage.

In [4]:
if MODEL_NAME not in {"logistic_regression", "random_forest"}:
    raise ValueError("MODEL_NAME must be 'logistic_regression' or 'random_forest'.")
if PREDICTION_SAMPLE_SIZE < 1:
    raise ValueError("PREDICTION_SAMPLE_SIZE must be a positive integer.")

labels_frame = read_labels_csv(LABELS_CSV)
metadata = load_data(seed=SEED)
if metadata is None:
    raise RuntimeError("SpecMine metadata loading failed; training cannot continue.")

training_features, training_labels, training_repositories = prepare_training_data(
    metadata,
    labels_frame,
    seed=SEED,
)
print(f"Labeled specs used for training/evaluation: {len(training_features)}")
print(f"Labeled repositories: {training_repositories.nunique()}")
print(f"Label counts: {training_labels.value_counts().sort_index().to_dict()}")

⠙ Attempting to connect to Hugging Face and load SpecMine dataset...

/var/folders/3j/5z92g6nn29n86z0mwbtybwc00000gn/T/ipykernel_185/3858096653.py:6: UserWarning: Conflicting labels for duplicate spec_id value(s) ['ebf86b98e601f589']; treating them as blank/ambiguous.
  labels_frame = read_labels_csv(LABELS_CSV)


INFO: Excluding 1 blank/ambiguous labels
INFO: Loading precomputed features for 97 labeled specs
INFO: Engineering features for 97 labeled specs
INFO: Commit history was not provided; volatility features skipped
INFO: Prepared 97 labeled specs from 64 repositories; supplied-label counts: {0: 69, 1: 28}


Labeled specs used for training/evaluation: 97
Labeled repositories: 64
Label counts: {0: 69, 1: 28}


## Evaluate on a repository-disjoint holdout

Choose one baseline by changing `MODEL_NAME` above. The reported metrics compare predictions to the supplied labels in the holdout only; they are not authorship-accuracy estimates unless those labels are verified ground truth.

In [5]:
training_results = train_models(
    training_features,
    training_labels,
    training_repositories,
    model_names=[MODEL_NAME],
    test_size=TEST_SIZE,
    seed=SEED,
)
holdout_result = training_results[MODEL_NAME]
print(
    f"{MODEL_NAME}: {holdout_result['n_train']} train specs from "
    f"{holdout_result['n_train_repositories']} repositories; "
    f"{holdout_result['n_test']} holdout specs from "
    f"{holdout_result['n_test_repositories']} repositories"
)
display(pd.DataFrame([holdout_result["metrics"]], index=[MODEL_NAME]))

INFO: Split: 83 training specs/51 repos, 14 holdout specs/13 repos; features: 45; labels: train={0: 58, 1: 25} holdout={0: 11, 1: 3}
INFO: Training logistic_regression
INFO: logistic_regression holdout supplied-label metrics: {'accuracy': 0.9285714285714286, 'precision': 0.75, 'recall': 1.0, 'f1': 0.8571428571428571, 'roc_auc': 0.9696969696969697, 'pr_auc': 0.9166666666666665, 'brier_score': 0.11138564246785468, 'ece': 0.2664629541321032, 'confusion_matrix': [[10, 1], [0, 3]], 'support': {'0': 11, '1': 3}}


logistic_regression: 83 train specs from 51 repositories; 14 holdout specs from 13 repositories


,accuracy,precision,recall,f1,roc_auc,pr_auc,brier_score,ece,confusion_matrix,support
logistic_regression,0.928571,0.75,1.0,0.857143,0.969697,0.916667,0.111386,0.266463,"[[10, 1], [0, 3]]","{'0': 11, '1': 3}"


## Refit on labeled data and predict unlabeled specs

The holdout model above is used only for evaluation. This cell clones its estimator and fits it on all nonblank labeled examples, then loads features only for a random sample of unlabeled specs. Every `spec_id` present in the CSV is excluded from that sample, including blank-label rows. The output contains predictions and positive-class probabilities, not ground-truth labels.

In [8]:
reviewed_ids = set(labels_frame[SPEC_ID_COLUMN].astype("string"))
metadata_ids = metadata[ID_COLUMN].astype("string")
prediction_pool = metadata.loc[~metadata_ids.isin(reviewed_ids)].dropna(
    subset=[ID_COLUMN, REPOSITORY_COLUMN]
)
prediction_pool = prediction_pool.drop_duplicates(subset=[ID_COLUMN])
if prediction_pool.empty:
    raise ValueError("No specs remain outside the IDs in the labels CSV.")

sample_count = min(PREDICTION_SAMPLE_SIZE, len(prediction_pool))
prediction_metadata = prediction_pool.sample(
    n=sample_count,
    random_state=SEED,
).reset_index(drop=True)
print(f"Random unlabeled specs selected: {len(prediction_metadata)}")

unlabeled_feature_rows = load_feature_data(prediction_metadata[[ID_COLUMN]])
if unlabeled_feature_rows is None or unlabeled_feature_rows.empty:
    raise ValueError("No precomputed features were found for the prediction sample.")
unlabeled_features = build_features(unlabeled_feature_rows)
feature_columns = holdout_result["feature_columns"]
missing_columns = [
    column for column in feature_columns if column not in unlabeled_features
]
if missing_columns:
    raise KeyError(f"Prediction features are missing model columns: {missing_columns}")

found_ids = set(unlabeled_features[ID_COLUMN].astype("string"))
missing_ids = set(prediction_metadata[ID_COLUMN].astype("string")) - found_ids
# if missing_ids:
#     raise ValueError(
#         f"Precomputed features are missing {len(missing_ids)} selected spec(s); "
#         "reduce the sample or check the feature dataset."
#     )
prediction_rows = prediction_metadata[[ID_COLUMN, REPOSITORY_COLUMN]].merge(
    unlabeled_features,
    on=ID_COLUMN,
    how="inner",
    validate="one_to_one",
)

training_numeric = (
    training_features[feature_columns]
    .apply(pd.to_numeric, errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
    .astype(float)
)
prediction_numeric = (
    prediction_rows[feature_columns]
    .apply(pd.to_numeric, errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
    .astype(float)
)
final_model = clone(holdout_result["model"])
final_model.fit(training_numeric, training_labels)
positive_class_index = list(final_model.classes_).index(1)
probability_ai = final_model.predict_proba(prediction_numeric)[:, positive_class_index]

predictions = pd.DataFrame(
    {
        SPEC_ID_COLUMN: prediction_rows[ID_COLUMN].astype("string"),
        REPOSITORY_COLUMN: prediction_rows[REPOSITORY_COLUMN],
        "predicted_label": (probability_ai >= 0.5).astype("int8"),
        "probability_ai_assisted": probability_ai,
    }
)
PREDICTIONS_CSV.parent.mkdir(parents=True, exist_ok=True)
predictions.to_csv(PREDICTIONS_CSV, index=False)
print(f"Saved {len(predictions)} predictions to {PREDICTIONS_CSV}")
display(predictions.head(20))

INFO: Commit history was not provided; volatility features skipped


Random unlabeled specs selected: 100
Saved 99 predictions to /Users/austinburt/Documents/Code/PythonEnv/Python/CS440/FinalProject-CS440/notebooks/unlabeled_predictions.csv


,spec_id,repo_name,predicted_label,probability_ai_assisted
0,376965b22a3905ca,harry18456/codebus,0,0.276396
1,387d30865b7f9de5,dnedrow/GitKit,0,0.433853
2,8cc24e4eb4e1dedb,moongchichi/chip,0,0.113900
3,0422c5e98e55348f,A-makarim/ai-platform-engineering,0,0.142901
4,d17c59d39033557e,fselich/dossier,0,0.254881
5,030e1c2d14166f00,MrMaxie/Orchidex,0,0.198761
6,f36abf898fe34999,kamwoh/yume,0,0.019803
7,729ec90be4c46af8,Benni-Math/ormstunga,0,0.448229
8,e03bb1de419f30b2,szzzzzz111/agentic-codeops,0,0.178744
9,282973d515d348b9,IndustriousKraken/octopus-autocoder,0,0.442051
